# 05 — Build Analysis Panel

This notebook merges the three cleaned datasets into a single analysis-ready panel for the
border-county difference-in-differences design (Dube, Lester & Reich 2010).

**Inputs:**
- `data/intermediate/qcew_panel.parquet` — county × year × industry employment/wage data
- `data/intermediate/min_wage_panel.parquet` — state × year minimum wages
- `data/intermediate/border_pairs.parquet` — 1,055 cross-state county pairs with min wage divergence

**Processing steps:**
1. Impute 2 missing state-years in the min wage panel (GA and WY — sub-federal states missing one January value).
2. Filter QCEW to border counties only.
3. Merge minimum wages onto each county × year row.
4. Attach pair IDs — each county can appear in multiple pairs.
5. Create log-transformed outcome variables.
6. Flag treated county in each pair × year (higher-wage state = treated).
7. Drop BLS-suppressed rows (`disclosure_code == 'N'`).
8. Save to `data/processed/analysis_panel.parquet`.

**Output:** `data/processed/analysis_panel.parquet`
- Unit of observation: county × year × industry × pair_id

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path().resolve().parent
INTERMEDIATE = ROOT / "data" / "intermediate"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUT = PROCESSED / "analysis_panel.parquet"

QCEW_FILE = INTERMEDIATE / "qcew_panel.parquet"
MIN_WAGE_FILE = INTERMEDIATE / "min_wage_panel.parquet"
PAIRS_FILE = INTERMEDIATE / "border_pairs.parquet"

FIRST_YEAR = 2010
LAST_YEAR = 2024

print("Inputs:")
for f in [QCEW_FILE, MIN_WAGE_FILE, PAIRS_FILE]:
    print(f"  {f.name:35s} {f.stat().st_size / 1e6:.2f} MB")
print(f"\nOutput: {OUTPUT}")

Inputs:
  qcew_panel.parquet                  3.00 MB
  min_wage_panel.parquet              0.00 MB
  border_pairs.parquet                0.02 MB

Output: /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/processed/analysis_panel.parquet


## 1. Load inputs

In [2]:
qcew = pd.read_parquet(QCEW_FILE)
mw = pd.read_parquet(MIN_WAGE_FILE)
pairs = pd.read_parquet(PAIRS_FILE)

print(f"QCEW panel     : {qcew.shape}")
print(f"Min wage panel : {mw.shape}")
print(f"Border pairs   : {pairs.shape}")
print()
print("QCEW columns  :", qcew.columns.tolist())
print("MW columns    :", mw.columns.tolist())
print("Pairs columns :", pairs.columns.tolist())

QCEW panel     : (215251, 13)
Min wage panel : (814, 4)
Border pairs   : (1055, 5)

QCEW columns  : ['area_fips', 'own_code', 'industry_code', 'year', 'disclosure_code', 'area_title', 'own_title', 'industry_title', 'annual_avg_estabs_count', 'annual_avg_emplvl', 'total_annual_wages', 'annual_avg_wkly_wage', 'avg_annual_pay']
MW columns    : ['state', 'state_fips', 'year', 'min_wage']
Pairs columns : ['pair_id', 'fips1', 'fips2', 'state1', 'state2']


## 2. Impute missing state-years in the min wage panel

GA and WY each have one missing January value in the FRED series. Both states have a statutory
minimum below the federal floor ($7.25), so the correct imputation is $7.25 — the rate employers
are legally required to pay.

In [3]:
FEDERAL_MIN = 7.25

# Build a complete state × year grid for 2009–2024
all_states = mw["state"].unique()
all_years = range(2009, LAST_YEAR + 1)
full_grid = pd.MultiIndex.from_product([all_states, all_years], names=["state", "year"])
full_grid = pd.DataFrame(index=full_grid).reset_index()

mw_complete = full_grid.merge(mw, on=["state", "year"], how="left")

# Identify missing state-years
missing = mw_complete[mw_complete["min_wage"].isna()]
print(f"Missing state-years before imputation: {len(missing)}")
print(missing[["state", "year"]])

# Impute state_fips for the missing rows
fips_lookup = mw.drop_duplicates("state").set_index("state")["state_fips"]
mw_complete["state_fips"] = mw_complete["state_fips"].fillna(
    mw_complete["state"].map(fips_lookup)
)

# Impute min_wage as federal floor
mw_complete["min_wage"] = mw_complete["min_wage"].fillna(FEDERAL_MIN)

print(
    f"\nMissing state-years after imputation : {mw_complete['min_wage'].isna().sum()}"
)
print(f"Total rows in completed panel        : {len(mw_complete):,}")

Missing state-years before imputation: 2
    state  year
175    GA  2024
815    WY  2024

Missing state-years after imputation : 0
Total rows in completed panel        : 816


## 3. Filter QCEW to border counties and analysis years

In [4]:
# All county FIPS that appear in at least one border pair
border_fips = set(pairs["fips1"]) | set(pairs["fips2"])
print(f"Unique border county FIPS : {len(border_fips):,}")

# Filter QCEW
qcew_border = qcew[
    qcew["area_fips"].isin(border_fips) & qcew["year"].between(FIRST_YEAR, LAST_YEAR)
].copy()

print(f"QCEW rows (all counties)  : {len(qcew):,}")
print(f"QCEW rows (border only)   : {len(qcew_border):,}")
print(f"Unique counties retained  : {qcew_border['area_fips'].nunique():,}")
print(f"Industries                : {qcew_border['industry_code'].unique().tolist()}")

Unique border county FIPS : 966
QCEW rows (all counties)  : 215,251
QCEW rows (border only)   : 56,931
Unique counties retained  : 966
Industries                : ['44-45', '72', '721', '722']


## 4. Merge minimum wages onto QCEW

Derive each county's state from the first 2 digits of its FIPS code, map to state abbreviation
using the Census FIPS table, then merge the annual minimum wage.

In [5]:
# Load FIPS → state abbreviation lookup
state_fips_df = pd.read_csv(
    ROOT / "data" / "raw" / "state_fips.txt",
    sep="|",
    dtype=str,
)[["STATE", "STUSAB"]].rename(columns={"STATE": "state_fips", "STUSAB": "state"})
state_fips_df["state_fips"] = state_fips_df["state_fips"].str.zfill(2)
fips_to_abbr = dict(zip(state_fips_df["state_fips"], state_fips_df["state"]))

# Derive state from county FIPS
qcew_border["state_fips"] = qcew_border["area_fips"].str[:2]
qcew_border["state"] = qcew_border["state_fips"].map(fips_to_abbr)

# Merge minimum wage
mw_analysis = mw_complete[mw_complete["year"].between(FIRST_YEAR, LAST_YEAR)]

qcew_mw = qcew_border.merge(
    mw_analysis[["state", "year", "min_wage"]],
    on=["state", "year"],
    how="left",
)

unmatched_mw = qcew_mw["min_wage"].isna().sum()
print(f"Rows after merging min wages : {len(qcew_mw):,}")
print(f"Rows with no min wage match  : {unmatched_mw:,}  (should be 0)")
qcew_mw[["area_fips", "state", "year", "industry_code", "min_wage"]].head()

Rows after merging min wages : 56,931
Rows with no min wage match  : 0  (should be 0)


,area_fips,state,year,industry_code,min_wage
0,01003,AL,2010,44-45,7.25
1,01003,AL,2011,44-45,7.25
2,01003,AL,2012,44-45,7.25
3,01003,AL,2013,44-45,7.25
4,01003,AL,2014,44-45,7.25


## 5. Attach pair IDs

Each border county may appear in multiple pairs (e.g. a corner county bordered by two different
states). We keep the panel long — one row per county × year × industry × pair.

In [6]:
# Explode pairs into two long tables — one per side of the pair
pairs_left = pairs[["pair_id", "fips1", "fips2", "state1", "state2"]].rename(
    columns={
        "fips1": "area_fips",
        "state1": "pair_own_state",
        "state2": "pair_other_state",
    }
)
pairs_right = pairs[["pair_id", "fips1", "fips2", "state1", "state2"]].rename(
    columns={
        "fips2": "area_fips",
        "state2": "pair_own_state",
        "state1": "pair_other_state",
    }
)

# Stack and keep relevant columns
pairs_long = pd.concat([pairs_left, pairs_right], ignore_index=True)[
    ["pair_id", "area_fips", "pair_own_state", "pair_other_state"]
]

# Merge onto QCEW
panel = qcew_mw.merge(pairs_long, on="area_fips", how="inner")

print(f"Rows before pair merge : {len(qcew_mw):,}")
print(
    f"Rows after pair merge  : {len(panel):,}  (more rows = counties in multiple pairs)"
)
print(f"Unique pair_ids        : {panel['pair_id'].nunique():,}")
print(f"Unique counties        : {panel['area_fips'].nunique():,}")

Rows before pair merge : 56,931
Rows after pair merge  : 124,590  (more rows = counties in multiple pairs)
Unique pair_ids        : 1,055
Unique counties        : 966


## 6. Create log-transformed outcome variables

Following Dube et al. (2010), we work in logs throughout:
- `log_emp` — log average employment (primary outcome)
- `log_wage` — log average weekly wage
- `log_estabs` — log establishment count
- `log_min_wage` — log minimum wage (the treatment variable)

We add 1 inside the log for employment and establishments to handle any zero counts without
dropping rows.

In [7]:
# Ensure numeric types
for col in [
    "annual_avg_emplvl",
    "annual_avg_wkly_wage",
    "annual_avg_estabs_count",
    "min_wage",
]:
    panel[col] = pd.to_numeric(panel[col], errors="coerce")

# Log transformations
panel["log_emp"] = np.log(panel["annual_avg_emplvl"] + 1)
panel["log_wage"] = np.log(panel["annual_avg_wkly_wage"].replace(0, np.nan))
panel["log_estabs"] = np.log(panel["annual_avg_estabs_count"] + 1)
panel["log_min_wage"] = np.log(panel["min_wage"])

print("Log variable summary:")
print(panel[["log_emp", "log_wage", "log_estabs", "log_min_wage"]].describe().round(3))

Log variable summary:
          log_emp    log_wage  log_estabs  log_min_wage
count  124590.000  104245.000  124590.000    124590.000
mean        5.697       5.915       3.927         2.101
std         3.013       0.376       1.666         0.211
min         0.000       3.989       0.000         1.639
25%         4.522       5.628       2.773         1.981
50%         6.303       5.922       3.807         1.988
75%         7.703       6.190       4.984         2.197
max        12.506       8.058       9.658         2.862


## 7. Flag treated county in each pair × year

Within each pair and year, the county in the **higher minimum wage state** is `treated = 1`.
When both states have the same wage in a given year, `treated = 0` for both (no variation that year).

In [8]:
# Merge the partner state's minimum wage for comparison
partner_mw = mw_analysis[["state", "year", "min_wage"]].rename(
    columns={"state": "pair_other_state", "min_wage": "partner_min_wage"}
)
panel = panel.merge(partner_mw, on=["pair_other_state", "year"], how="left")

# treated = 1 if this county's state has a strictly higher minimum wage than its partner
panel["treated"] = (panel["min_wage"] > panel["partner_min_wage"]).astype(int)

print("Treated distribution across pair × year × county × industry rows:")
print(panel["treated"].value_counts())
print(f"\nTreatment rate: {panel['treated'].mean():.1%}")

Treated distribution across pair × year × county × industry rows:
treated
0    71246
1    53344
Name: count, dtype: int64

Treatment rate: 42.8%


## 8. Drop BLS-suppressed rows

In [9]:
n_before = len(panel)
suppressed = (panel["disclosure_code"] == "N").sum()
print(f"Rows before suppression drop : {n_before:,}")
print(f"Suppressed rows (code = 'N') : {suppressed:,}  ({suppressed/n_before:.1%})")

panel = panel[panel["disclosure_code"] != "N"].copy()
print(f"Rows after suppression drop  : {len(panel):,}")

Rows before suppression drop : 124,590
Suppressed rows (code = 'N') : 20,345  (16.3%)
Rows after suppression drop  : 104,245


## 9. Final panel — select and order columns

In [10]:
FINAL_COLS = [
    # Identifiers
    "pair_id",
    "area_fips",
    "state",
    "year",
    "industry_code",
    "industry_title",
    # Treatment
    "treated",
    "min_wage",
    "partner_min_wage",
    "log_min_wage",
    # Outcomes (raw)
    "annual_avg_emplvl",
    "annual_avg_wkly_wage",
    "annual_avg_estabs_count",
    "total_annual_wages",
    "avg_annual_pay",
    # Outcomes (log)
    "log_emp",
    "log_wage",
    "log_estabs",
    # Metadata
    "area_title",
]

panel_final = (
    panel[FINAL_COLS]
    .sort_values(["pair_id", "area_fips", "industry_code", "year"])
    .reset_index(drop=True)
)

print("Final panel shape:", panel_final.shape)
print("\nMissing values:")
print(panel_final.isnull().sum()[panel_final.isnull().sum() > 0])
panel_final.head(10)

Final panel shape: (104245, 19)

Missing values:
Series([], dtype: int64)


,pair_id,area_fips,state,year,industry_code,industry_title,treated,min_wage,partner_min_wage,log_min_wage,annual_avg_emplvl,annual_avg_wkly_wage,annual_avg_estabs_count,total_annual_wages,avg_annual_pay,log_emp,log_wage,log_estabs,area_title
0,1,01003,AL,2010,44-45,Retail trade,0,7.25,7.25,1.981001,11909,446,965,276147765,23188,9.385134,6.100319,6.873164,"Baldwin County, Alabama"
1,1,01003,AL,2011,44-45,Retail trade,0,7.25,7.25,1.981001,12421,454,963,292954393,23586,9.427224,6.118097,6.871091,"Baldwin County, Alabama"
2,1,01003,AL,2012,44-45,Retail trade,0,7.25,7.67,1.981001,12518,463,956,301354325,24073,9.435003,6.137727,6.863803,"Baldwin County, Alabama"
3,1,01003,AL,2013,44-45,Retail trade,0,7.25,7.79,1.981001,12642,471,963,309634817,24493,9.444859,6.154858,6.871091,"Baldwin County, Alabama"
4,1,01003,AL,2014,44-45,Retail trade,0,7.25,7.93,1.981001,13183,482,989,330697194,25085,9.486759,6.177944,6.897705,"Baldwin County, Alabama"
5,1,01003,AL,2015,44-45,Retail trade,0,7.25,8.05,1.981001,13568,497,998,350607522,25841,9.515543,6.208590,6.906755,"Baldwin County, Alabama"
6,1,01003,AL,2016,44-45,NAICS 44-45 Retail trade,0,7.25,8.05,1.981001,13717,507,1020,361405392,26347,9.526464,6.228511,6.928538,"Baldwin County, Alabama"
7,1,01003,AL,2017,44-45,NAICS 44-45 Retail trade,0,7.25,8.10,1.981001,13681,519,1035,369474976,27007,9.523836,6.251904,6.943122,"Baldwin County, Alabama"
8,1,01003,AL,2018,44-45,NAICS 44-45 Retail trade,0,7.25,8.25,1.981001,13549,543,1045,382853922,28257,9.514142,6.297109,6.952729,"Baldwin County, Alabama"
9,1,01003,AL,2019,44-45,NAICS 44-45 Retail trade,0,7.25,8.46,1.981001,13968,562,1038,408001066,29209,9.544596,6.331502,6.946014,"Baldwin County, Alabama"


## 10. Summary statistics

In [11]:
print("=" * 55)
print("PANEL SUMMARY")
print("=" * 55)
print(f"Total rows                   : {len(panel_final):,}")
print(f"Unique pair_ids              : {panel_final['pair_id'].nunique():,}")
print(f"Unique counties              : {panel_final['area_fips'].nunique():,}")
print(f"Unique states                : {panel_final['state'].nunique()}")
print(f"Years                        : {FIRST_YEAR}–{LAST_YEAR}")
print(f"Industries                   : {panel_final['industry_code'].nunique()}")
print(f"Treatment rate               : {panel_final['treated'].mean():.1%}")
print()
print("Rows by industry:")
print(
    panel_final.groupby(["industry_code", "industry_title"])
    .size()
    .reset_index(name="n_rows")
    .to_string(index=False)
)
print()
print("Mean log employment by industry:")
print(panel_final.groupby("industry_code")["log_emp"].mean().round(3).to_string())

PANEL SUMMARY
Total rows                   : 104,245
Unique pair_ids              : 1,055
Unique counties              : 961
Unique states                : 49
Years                        : 2010–2024
Industries                   : 4
Treatment rate               : 43.6%

Rows by industry:
industry_code                              industry_title  n_rows
        44-45                    NAICS 44-45 Retail trade   18691
        44-45                                Retail trade   12491
           72             Accommodation and food services   10176
           72    NAICS 72 Accommodation and food services   14889
          721                               Accommodation    8716
          721                     NAICS 721 Accommodation   12919
          722           Food services and drinking places   10615
          722 NAICS 722 Food services and drinking places   15748

Mean log employment by industry:
industry_code
44-45    7.160
72       7.270
721      5.618
722      6.933


## 11. Save to parquet

In [12]:
panel_final.to_parquet(OUTPUT, index=False)

print(f"Saved to  : {OUTPUT}")
print(f"File size : {OUTPUT.stat().st_size / 1e6:.2f} MB")
print(f"Rows      : {len(panel_final):,}")

# Read back to verify
check = pd.read_parquet(OUTPUT)
print(f"\nRead back : {check.shape} — OK ✓")
check.head()

Saved to  : /Users/ogechukwuezenwa/Documents/Duke_University_Projects/Spring_2026/IDS_701-Causal-Inference_Final/data/processed/analysis_panel.parquet
File size : 1.85 MB
Rows      : 104,245

Read back : (104245, 19) — OK ✓


,pair_id,area_fips,state,year,industry_code,industry_title,treated,min_wage,partner_min_wage,log_min_wage,annual_avg_emplvl,annual_avg_wkly_wage,annual_avg_estabs_count,total_annual_wages,avg_annual_pay,log_emp,log_wage,log_estabs,area_title
0,1,01003,AL,2010,44-45,Retail trade,0,7.25,7.25,1.981001,11909,446,965,276147765,23188,9.385134,6.100319,6.873164,"Baldwin County, Alabama"
1,1,01003,AL,2011,44-45,Retail trade,0,7.25,7.25,1.981001,12421,454,963,292954393,23586,9.427224,6.118097,6.871091,"Baldwin County, Alabama"
2,1,01003,AL,2012,44-45,Retail trade,0,7.25,7.67,1.981001,12518,463,956,301354325,24073,9.435003,6.137727,6.863803,"Baldwin County, Alabama"
3,1,01003,AL,2013,44-45,Retail trade,0,7.25,7.79,1.981001,12642,471,963,309634817,24493,9.444859,6.154858,6.871091,"Baldwin County, Alabama"
4,1,01003,AL,2014,44-45,Retail trade,0,7.25,7.93,1.981001,13183,482,989,330697194,25085,9.486759,6.177944,6.897705,"Baldwin County, Alabama"


## 12. Findings & Notes

### Results

| Metric | Value |
|--------|-------|
| Final panel rows | 104,245 |
| Final panel columns | 19 |
| File size | 1.85 MB |
| Unique border pairs | 1,055 |
| Unique counties | 961 |
| Unique states | 49 |
| Years | 2010–2024 |
| Industries | 4 |
| Missing values | 0 |
| Treatment rate | 43.6% |

### Pipeline row counts
| Step | Rows |
|------|------|
| QCEW full panel | 215,251 |
| After filtering to border counties | 56,931 |
| After attaching pair IDs (counties in multiple pairs) | 124,590 |
| BLS-suppressed rows dropped (16.3%) | −20,345 |
| **Final analysis panel** | **104,245** |

### Rows by industry
| Industry code | Title | Rows |
|---------------|-------|------|
| 722 | Food services and drinking places | 26,363 |
| 44-45 | Retail trade | 31,182 |
| 72 | Accommodation and food services | 25,065 |
| 721 | Accommodation | 21,635 |

### Mean log employment by industry
| Industry | Mean log emp |
|----------|-------------|
| 72 (Accommodation & food) | 7.270 |
| 44-45 (Retail) | 7.160 |
| 722 (Food services) | 6.933 |
| 721 (Accommodation) | 5.618 |

### Key observations

- **BLS suppression (16.3%):** 20,345 rows were dropped because BLS withheld data for confidentiality. This is expected — small counties with few establishments in a given industry often get suppressed. These rows are concentrated in NAICS 721 (Accommodation) and 722 (Food services) in rural border counties.

- **Treatment rate (43.6%):** Slightly below 50% because not all pairs have diverging wages in every year — when both states are at the federal floor in a given year, both counties get `treated = 0`.

- **County count (961 vs 966):** 5 border counties present in `border_pairs` had no QCEW data for any industry in 2010–2024 and dropped out during the inner merge.

- **Industry title inconsistency:** `industry_title` has two naming formats for the same code (e.g. `"Retail trade"` vs `"NAICS 44-45 Retail trade"`) because BLS changed its title convention around 2015–2016. This does not affect analysis — always filter by `industry_code`, not `industry_title`.

- **Imputation:** GA and WY were both missing their 2024 January minimum wage in the FRED series. Both states hold sub-federal statutory minimums ($5.15), so the missing value was correctly imputed as the federal floor of $7.25.